<a href="https://colab.research.google.com/github/omarhatem44/Flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarhatem44/Flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

#**Setup Cell**

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Ranking, driven by a probability classifier.** The decision is a *ranking* — order pages so an editor reviews the most promising first. I implement it as **binary classification** under the hood: predict each page's probability of being in the "declining / needs review" class, then rank pages by that probability. So the model is a scorer; the deliverable is a ranked queue. Naming it this way fixes the right tools: probability outputs, top-K decisions, and ranking metrics rather than plain accuracy.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Task type: ranking via a binary probability classifier (score -> rank).")

Task type: ranking via a binary probability classifier (score -> rank).


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Proxy (starting point):** `is_declining = (trend_direction == "down")` — the starter label. It's a *defined rule*, not an observed future outcome: a bucket computed from the current window, so it can only say "looks down now."

**Stronger target (capstone direction):** a future observed outcome — features from the prior 90 days predicting decline over the *next* 30 days (`features(prior 90d) → decline(next 30d)`). This is leakage-safe because the feature and target windows don't overlap. I start with the proxy to get the workflow running, then move to the future-window label on the warehouse data.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Proxy label rule: is_declining = (trend_direction == 'down')")
print(df["trend_direction"].value_counts().to_string())

Proxy label rule: is_declining = (trend_direction == 'down')
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152


## 3. Success metric

*One metric you can defend. What number means 'good'?*


**Precision@K** (e.g. Precision@50). The editor works top-down through the queue, so what matters is: of the top K pages surfaced, how many actually needed review? Overall accuracy is the wrong target — a high average can still put junk in the top 50.

**The bar to beat** (verified starter numbers): baseline hand-rule Precision@50 = 0.240; random forest = 0.740. "Good" = clearly beating the transparent rule at the top of the list. I'd also watch **calibration** (a predicted 0.7 should be right ~70% of the time), since the score drives a human decision.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Primary metric: Precision@50 (top-K matches how the queue is used).")
print("Bar to beat -> baseline: 0.240   random forest: 0.740")


Primary metric: Precision@50 (top-K matches how the queue is used).
Bar to beat -> baseline: 0.240   random forest: 0.740


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one **page** (`content_id`). Below I load the slice, confirm the grain is unique per page, show the actual rows, and sketch the proxy target column.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Shape (rows, cols):", df.shape)
print("content_id unique per row?", df["content_id"].nunique() == len(df))

cols = ["content_id", "client_id", "impressions_90d", "avg_position", "ctr", "trend_direction"]
display(df[cols].head())

df["is_declining"] = (df["trend_direction"] == "down").astype(int)
print("\nProxy target distribution (is_declining):")
print(df["is_declining"].value_counts(normalize=True).round(3).to_string())

Shape (rows, cols): (30000, 44)
content_id unique per row? True


,content_id,client_id,impressions_90d,avg_position,ctr,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,10.6,0.76,down
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,0.05,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,0.09,down
3,content_331d6c4de07b,client_19581e27de,11751,6.2,0.49,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,44.0,0.13,down



Proxy target distribution (is_declining):
is_declining
1    0.542
0    0.458


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

~54% of pages are flagged declining and ~44% are declining *with* real demand. So a fixed rule like "review everything that's down" returns tens of thousands of pages — it flags but can't **prioritize**, which is useless against a team that reviews ~50 pages.

The pattern is too messy for an if-statement because "worth reviewing first" depends on many signals *at once* — impressions, position, CTR, age, engagement — interacting in ways no hand-tuned threshold captures. A learned model produces a continuous, comparable score that ranks pages *within* the declining pool. Same data, better ordering: Precision@50 goes 0.24 → 0.74. **Action:** the ranked queue plus reason codes tells an editor which pages to open first and why, so limited hours go to the pages most likely to need work. The model prioritizes; the human decides.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
declining = (df["trend_direction"] == "down").sum()
actionable = ((df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)).sum()
print(f"Declining pages: {declining:,} ({declining/len(df):.1%}) -> too many for a rule to prioritize")
print(f"Declining with demand: {actionable:,} ({actionable/len(df):.1%})")
print("A learned score ranks within this pool: Precision@50 0.24 -> 0.74")

Declining pages: 16,262 (54.2%) -> too many for a rule to prioritize
Declining with demand: 13,152 (43.8%)
A learned score ranks within this pool: Precision@50 0.24 -> 0.74


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.